# ModernBERT-base fine-tuning on GLUE/QNLI

Mục tiêu: fine-tune `answerdotai/ModernBERT-base` trên GLUE/QNLI và evaluate trên validation/dev set để đối chiếu với Table 5 của paper ModernBERT.

Theo Table 6 cho **ModernBERT-base / QNLI**:

- Learning rate: `8e-5`
- Weight decay: `5e-6`
- Epochs: `2`

Target tham khảo ở Table 5: **QNLI dev accuracy ≈ 93.9**.


In [1]:
# ============================================================
# 1. Install / upgrade libraries
# ============================================================
# Kaggle thường đã có sẵn transformers/datasets, nhưng nên upgrade để hỗ trợ ModernBERT ổn định.
# Sau khi chạy cell này, nếu Kaggle yêu cầu restart session thì restart rồi chạy lại từ đầu.

!pip -q install -U "transformers>=4.48.0" "datasets>=3.0.0" "accelerate>=0.34.0" evaluate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 103.2 MB/s eta 0:00:00


In [2]:
# ============================================================
# 2. Imports and config
# ============================================================
import os
import gc
import json
import random
import inspect
from pathlib import Path

import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

print("torch:", torch.__version__)
import transformers, datasets
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)

MODEL_NAME = "answerdotai/ModernBERT-base"
DATASET_NAME = "nyu-mll/glue"
TASK_NAME = "qnli"
OUTPUT_DIR = "./modernbert-base-qnli"

# Paper Table 6: ModernBERT-base / QNLI
LR = 8e-5
WEIGHT_DECAY = 5e-6
NUM_EPOCHS = 2

# Batch size không được paper ghi rõ. Chỉnh theo VRAM Kaggle.
# T4 thường ổn với 16; nếu OOM thì giảm xuống 8 và tăng gradient_accumulation_steps.
PER_DEVICE_TRAIN_BATCH_SIZE = 32
PER_DEVICE_EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1

MAX_LENGTH = 256
SEED = 42

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


torch: 2.10.0+cu128
transformers: 5.9.0
datasets: 4.8.5
CUDA available: True
GPU: Tesla T4


In [3]:
# ============================================================
# 3. Load GLUE/QNLI dataset
# ============================================================
# QNLI fields: question, sentence, label
# Splits: train, validation, test. Để tái hiện Table 5, dùng validation/dev, không dùng test.

raw_datasets = load_dataset(DATASET_NAME, TASK_NAME)

print(raw_datasets)
print("Train example:", raw_datasets["train"][0])
print("Validation example:", raw_datasets["validation"][0])
print("Labels:", raw_datasets["train"].features["label"].names)


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

qnli/train-00000-of-00001.parquet:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

qnli/validation-00000-of-00001.parquet:   0%|          | 0.00/872k [00:00<?, ?B/s]

qnli/test-00000-of-00001.parquet:   0%|          | 0.00/877k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/104743 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5463 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5463 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
})
Train example: {'question': 'When did the third Digimon series begin?', 'sentence': 'Unlike the two seasons before it and most of the seasons that followed, Digimon Tamers takes a darker and more realistic approach to its story featuring Digimon who do not reincarnate after their deaths and more complex character development in the original Japanese.', 'label': 1, 'idx': 0}
Validation example: {'question': 'What came into force after the new constitution was herald?', 'sentence': 'As of that day, the new constitution heralding the Second Republic came into force.', 'label': 0, 'idx': 0}
Labels: ['entailment', 'not_entailment']


In [4]:
# ============================================================
# 4. Tokenizer and preprocessing
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)

sentence1_key = "question"
sentence2_key = "sentence"

label_names = raw_datasets["train"].features["label"].names
num_labels = len(label_names)
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

print("id2label:", id2label)
print("label2id:", label2id)

def preprocess_function(examples):
    return tokenizer(
        examples[sentence1_key],
        examples[sentence2_key],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=[sentence1_key, sentence2_key, "idx"],
)

# Trainer expects label column named labels in some versions; label also usually works.
# Rename explicitly for compatibility.
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_datasets)
print(tokenized_datasets["train"][0].keys())


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

id2label: {0: 'entailment', 1: 'not_entailment'}
label2id: {'entailment': 0, 'not_entailment': 1}


Map:   0%|          | 0/104743 [00:00<?, ? examples/s]

Map:   0%|          | 0/5463 [00:00<?, ? examples/s]

Map:   0%|          | 0/5463 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5463
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5463
    })
})
dict_keys(['labels', 'input_ids', 'attention_mask'])


In [5]:
# ============================================================
# 5. Metric: QNLI uses accuracy
# ============================================================

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


In [6]:
# ============================================================
# 6. Load ModernBERT-base model
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    trust_remote_code=True,
)

# Disable cache for training if present.
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

print(model.config)


model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ModernBertConfig {
  "architectures": [
    "ModernBertForMaskedLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 50281,
  "classifier_activation": "gelu",
  "classifier_bias": false,
  "classifier_dropout": 0.0,
  "classifier_pooling": "mean",
  "cls_token_id": 50281,
  "decoder_bias": true,
  "deterministic_flash_attn": false,
  "dtype": "float32",
  "embedding_dropout": 0.0,
  "eos_token_id": 50282,
  "global_attn_every_n_layers": 3,
  "gradient_checkpointing": false,
  "hidden_activation": "gelu",
  "hidden_size": 768,
  "id2label": {
    "0": "entailment",
    "1": "not_entailment"
  },
  "initializer_cutoff_factor": 2.0,
  "initializer_range": 0.02,
  "intermediate_size": 1152,
  "label2id": {
    "entailment": 0,
    "not_entailment": 1
  },
  "layer_norm_eps": 1e-05,
  "layer_types": [
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "sliding_attention",
    "full_attention

In [7]:
# ============================================================
# 7. TrainingArguments
# ============================================================
# Cell này viết theo kiểu tương thích nhiều version transformers:
# - bản mới dùng eval_strategy
# - bản cũ dùng evaluation_strategy

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16

args_signature = inspect.signature(TrainingArguments.__init__)
args_params = set(args_signature.parameters.keys())

def build_training_args():
    kwargs = dict(
        output_dir=OUTPUT_DIR,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        logging_steps=100,
        save_steps=1000,
        eval_steps=1000,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        fp16=use_fp16,
        bf16=use_bf16,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
    )

    # Strategy argument name depends on transformers version.
    if "eval_strategy" in args_params:
        kwargs["eval_strategy"] = "steps"
    else:
        kwargs["evaluation_strategy"] = "steps"

    if "save_strategy" in args_params:
        kwargs["save_strategy"] = "steps"

    # Remove unsupported kwargs for very old/new versions.
    kwargs = {k: v for k, v in kwargs.items() if k in args_params}
    return TrainingArguments(**kwargs)

training_args = build_training_args()
print(training_args)
print("use_fp16:", use_fp16, "use_bf16:", use_bf16)


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1000,
eval_strategy=IntervalStrategy.STEPS,
eval_u

In [8]:
# ============================================================
# 8. Trainer and fine-tuning
# ============================================================
# Paper dùng early stopping cho fine-tuning runs.
# Với QNLI epochs=2, patience=2 là an toàn; không dừng quá sớm.

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# tokenizer= bị bỏ ở một số version; processing_class= có ở version mới.
trainer_signature = inspect.signature(Trainer.__init__)
trainer_params = set(trainer_signature.parameters.keys())
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

train_result = trainer.train()
print(train_result)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Accuracy
1000,0.475371,0.411136,0.918543
2000,0.195490,0.411213,0.930624
3000,0.170661,0.405832,0.930807
3274,0.170649,0.398281,0.934651


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3274, training_loss=0.36519587236566375, metrics={'train_runtime': 3639.341, 'train_samples_per_second': 57.562, 'train_steps_per_second': 0.9, 'total_flos': 1.6227918887331984e+16, 'train_loss': 0.36519587236566375, 'epoch': 2.0})


In [9]:
# ============================================================
# 9. Evaluate on QNLI validation/dev set
# ============================================================

eval_results = trainer.evaluate(tokenized_datasets["validation"])
print(eval_results)

acc = eval_results["eval_accuracy"] * 100
print(f"QNLI validation/dev accuracy: {acc:.2f}")
print("Paper Table 5 ModernBERT-base QNLI target: about 93.9")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with open(Path(OUTPUT_DIR) / "eval_results_qnli.json", "w") as f:
    json.dump(eval_results, f, indent=2)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Step,Accuracy
0.170649,0.398281,3274,0.934651


{'eval_loss': 0.3982812762260437, 'eval_accuracy': 0.9346512904997254}
QNLI validation/dev accuracy: 93.47
Paper Table 5 ModernBERT-base QNLI target: about 93.9


In [10]:
# ============================================================
# 10. Save best model and tokenizer
# ============================================================

best_dir = "./modernbert-base-qnli-best"
trainer.save_model(best_dir)
tokenizer.save_pretrained(best_dir)

print("Best checkpoint saved to:", best_dir)
print("Trainer best model checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best checkpoint saved to: ./modernbert-base-qnli-best
Trainer best model checkpoint: ./modernbert-base-qnli/checkpoint-3274
Best metric: 0.9346512904997254


## Ghi chú nếu điểm chưa gần Table 5

Nếu accuracy thấp hơn rõ rệt, kiểm tra các điểm sau:

1. Kaggle có đang bật GPU không.
2. `transformers` đã đủ mới chưa.
3. Có bị OOM âm thầm rồi giảm batch quá nhỏ không.
4. Chạy lại với seed khác, ví dụ `SEED = 123` hoặc `SEED = 3407`.
5. Không evaluate trên `test`; Table 5 là GLUE dev/validation score.
